Loading the data

In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

sample = pd.read_pickle("../artifacts/sample.pkl")
emb = np.load("../artifacts/embeddings.npy")
assert len(sample) == len(emb), "sample and embeddings are out of sync!"
print(sample.shape, emb.shape)

(50001, 14) (50001, 384)


Choosing the number of topics (k)

In [2]:
rng = np.random.default_rng(0)
idx = rng.choice(len(emb), size=8000, replace=False)   # subsample for speed

for k in [10, 15, 20, 25, 30]:
    labels_k = KMeans(n_clusters=k, n_init=3, random_state=0).fit_predict(emb[idx])
    print(k, round(silhouette_score(emb[idx], labels_k, metric="cosine"), 4))

10 0.0628
15 0.0686
20 0.0688
25 0.0579
30 0.0587


Clustering all the reviews

In [3]:
K = 20   # adjust based on Cell 2 and on how readable the topics are
km = KMeans(n_clusters=K, n_init=5, random_state=42).fit(emb)
sample["topic_id"] = km.labels_

centroids = km.cluster_centers_ / np.linalg.norm(km.cluster_centers_, axis=1, keepdims=True)
sample.topic_id.value_counts().sort_index()

topic_id
0     1760
1     1613
2     3715
3     3928
4     1569
5     2753
6     1363
7     2859
8     1191
9     1729
10    2593
11    4834
12    3610
13    2171
14    1072
15    1993
16    3239
17    2054
18    3967
19    1988
Name: count, dtype: int64

Finding the keywords for each topic with c-TF-IDF

In [11]:
DOMAIN_STOP = {"br", "product", "products", "just", "like", "good", "great", "really", "would",
               "one", "get", "got", "also", "much", "even", "make", "made", "use", "used", "buy",
               "bought", "try", "tried", "love", "best", "well", "think", "know", "lot", "little",
               "bit", "time", "ve", "don", "didn", "doesn", "amazon", "taste", "tastes", "flavor", "flavors", "delicious", "tasty", "stuff", "better"}
STOP = list(ENGLISH_STOP_WORDS | DOMAIN_STOP)

def ctfidf_keywords(texts, labels, top_n=10):
    # 1. one big "document" per cluster
    docs = pd.Series(texts).groupby(labels).apply(" ".join)
    cv = CountVectorizer(stop_words=STOP, ngram_range=(1, 2), min_df=5,
                         max_features=40000, token_pattern=r"(?u)\b[a-zA-Z]{2,}\b")
    tf = cv.fit_transform(docs.values).astype(float)
    # 2. term frequency within each cluster
    tf = tf.multiply(1 / tf.sum(axis=1)).tocsr()
    # 3. idf: penalise words common to all clusters
    idf = np.log(1 + tf.shape[0] / np.asarray((tf > 0).sum(axis=0)).ravel())
    scores = tf.multiply(idf).tocsr()
    vocab = np.array(cv.get_feature_names_out())
    out = {}
    for row, lab in enumerate(docs.index):
        r = scores[row].toarray().ravel()
        words = []
        for w in vocab[np.argsort(-r)]:
            # skip near-duplicates like "dog" + "dog food"
            if any(w in x.split() or x in w.split() for x in words):
                continue
            words.append(w)
            if len(words) == top_n:
                break
        out[int(lab)] = words
    return out

keywords = ctfidf_keywords(sample.text_for_embedding.values, sample.topic_id.values)
for t, w in keywords.items():
    print(t, "|", ", ".join(w))

0 | pasta, soup, noodles, chicken, sauce, cheese, add, noodle, rice, eat
1 | price, store, shipping, box, local, deal, pack, expensive, cost, grocery
2 | drink, water, sugar, sweet, juice, soda, syrup, coconut, drinks, bottle
3 | price, popcorn, quality, excellent, food, mix, sweet, eat, bad, fresh
4 | gluten, bread, free, mix, flour, baking, cake, wheat, pancakes, gf
5 | cookies, candy, cookie, chocolate, gum, sweet, sugar, box, soft, hard
6 | cat, food, cats, eat, litter, kitty, chicken, dry, loves, fancy feast
7 | salt, price, order, food, popcorn, store, fresh, organic, eat, favorite
8 | food, dog, dogs, eat, loves, chicken, puppy, ingredients, bag, foods
9 | chocolate, dark, cocoa, bar, bars, milk, hot, sweet, sugar, chocolates
10 | box, bag, order, ordered, bags, received, package, packaging, arrived, shipping
11 | coffee, cup, cups, roast, decaf, coffees, starbucks, price, drink, espresso
12 | tea, teas, green, drink, cup, iced, bags, earl, water, bag
13 | dog, dogs, treats, lov

Summarizing table per topic

In [12]:
summary = sample.groupby("topic_id").agg(
    n=("Score", "size"),
    avg_score=("Score", "mean"),
    pct_negative=("Score", lambda s: (s <= 2).mean()),
).round(3)
summary["share"] = (summary.n / len(sample)).round(3)
summary["keywords"] = summary.index.map(lambda t: ", ".join(keywords[t][:6]))
summary.sort_values("n", ascending=False)

,n,avg_score,pct_negative,share,keywords
topic_id,,,,,
11,4834,4.165,0.139,0.097,"coffee, cup, cups, roast, decaf, coffees"
18,3967,4.432,0.080,0.079,"snack, bars, chips, eat, healthy, bag"
3,3928,4.140,0.145,0.079,"price, popcorn, quality, excellent, food, mix"
2,3715,4.028,0.172,0.074,"drink, water, sugar, sweet, juice, soda"
12,3610,4.383,0.100,0.072,"tea, teas, green, drink, cup, iced"
16,3239,4.439,0.088,0.065,"sauce, hot, chicken, salt, spicy, seasoning"
7,2859,4.356,0.112,0.057,"salt, price, order, food, popcorn, store"
5,2753,4.248,0.126,0.055,"cookies, candy, cookie, chocolate, gum, sweet"
10,2593,3.052,0.426,0.052,"box, bag, order, ordered, bags, received"


Reading example reviews from each topic

In [13]:
def examples(topic, n=3):
    idx = np.where(sample.topic_id == topic)[0]
    sims = emb[idx] @ centroids[topic]
    best = idx[np.argsort(-sims)[:n]]
    for _, r in sample.iloc[best].iterrows():
        print(f"  {r.Score}★ {r.Summary} — {r.Text[:180]}")

for t in range(K):
    print(f"\n=== Topic {t}: {', '.join(keywords[t][:5])}")
    examples(t)


=== Topic 0: pasta, soup, noodles, chicken, sauce
  5★ Wonderful! — This is one of the best food products available. It's a healthy alternative to traditional pastas and has wonderful texture and most of all - tastes great!
  5★ Great Pasta — Our family has eaten this brand for years, and love it. This flavor in particular is my favorite.
  5★ Awesome product!!! — I absolutely love this pasta!!! It tastes great, cooks in no time at all, and I would order again and again and again! I could eat this all day long! I like to boil them in a littl

=== Topic 1: price, store, shipping, box, local
  2★ Overpriced!!! Even in Amazon!!!!!! — The product is of excellent quality and of great taste and is imported from India. Unlike most of the items available in Amazon.com, this one is overtly priced @$23.99 when you can
  5★ Great product, but can be cheaper. — Excellent stock, the best flavor with the least additives I have seen. However, the price of $32 for 6 or $5.33 each is over double the p

Naming the topics myself

In [14]:

labels = {
    0:  "Pasta, soups & noodles",
    1:  "Price & value vs. stores",
    2:  "Drinks, juices & sweeteners",
    3:  "General taste & quality",
    4:  "Gluten-free & baking",
    5:  "Cookies, candy & gum",
    6:  "Cat food",
    7:  "Popcorn & salty snacks",
    8:  "Dog food",
    9:  "Chocolate & cocoa",
    10: "Packaging & delivery",
    11: "Coffee & K-cups",
    12: "Tea",
    13: "Dog treats & chews",
    14: "Coconut/olive oil & personal care",
    15: "Baby formula & milk",
    16: "Hot sauces & seasonings",
    17: "Cereal, oatmeal & nut butters",
    18: "Healthy snacks, bars & chips",
    19: "Gifts & orders",
}

missing = set(range(K)) - set(labels)
print("Still to label:", missing)

Still to label: set()


Saving the results

In [10]:
sample["topic_label"] = sample.topic_id.map(labels).fillna("Unlabelled")
sample.to_pickle("../artifacts/reviews_topics.pkl")

topics = [{"id": int(t), "label": labels.get(t, f"Topic {t}"), "keywords": keywords[t],
           "n": int(summary.loc[t, "n"]), "share": float(summary.loc[t, "share"]),
           "avg_score": float(summary.loc[t, "avg_score"]),
           "pct_negative": float(summary.loc[t, "pct_negative"])} for t in range(K)]
with open("../artifacts/topics.json", "w") as f:
    json.dump(topics, f, indent=2)
np.save("../artifacts/topic_centroids.npy", centroids)
print("Saved reviews_topics.pkl, topics.json, topic_centroids.npy")

Saved reviews_topics.pkl, topics.json, topic_centroids.npy
